In [0]:
# Imports and Configuration

from pyspark.sql import functions as F
import requests

VOLUME_PATH = "/Volumes/workspace/default/senior_de_assessment"
OUTPUT_PATH = f"{VOLUME_PATH}/output"

sales_path = f"{VOLUME_PATH}/sales_data 2.csv"
product_path = f"{VOLUME_PATH}/product_reference 2.csv"
API_URL = "https://api.exchangerate-api.com/v4/latest/EUR"

In [0]:
# Read Source Data

sales_raw_df = spark.read.option("header", True).option("inferSchema", False).csv(sales_path)
product_raw_df = spark.read.option("header", True).option("inferSchema", False).csv(product_path)

print("Sales records:", sales_raw_df.count())
print("Product records:", product_raw_df.count())

In [0]:
# Remove Duplicates

sales_df = sales_raw_df.dropDuplicates(["OrderID"])

print("Records after deduplication:", sales_df.count())

In [0]:
# Type Conversion

sales_df = sales_df.withColumn("SaleAmount", F.col("SaleAmount").cast("double"))
sales_df = sales_df.withColumn("Discount", F.coalesce(F.col("Discount").cast("double"), F.lit(0.0)))
sales_df = sales_df.withColumn("OrderDate", F.expr("try_to_date(replace(OrderDate, '/', '-'), 'dd-MM-yyyy')"))

display(sales_df)

In [0]:
# Product Lookup & Validation

product_lookup_df = product_raw_df.select("ProductID", "ProductName", "Category").dropDuplicates(["ProductID"])

sales_enriched_df = sales_df.join(product_lookup_df, "ProductID", "left")

sales_enriched_df = sales_enriched_df.withColumn("ValidationError",
    F.when(F.col("SaleAmount").isNull(), "INVALID_SALE_AMOUNT")
     .when(F.col("SaleAmount") <= 0, "INVALID_SALE_AMOUNT")
     .when(F.col("OrderDate").isNull(), "INVALID_ORDER_DATE")
     .when(F.col("CustomerID").isNull() | (F.trim(F.col("CustomerID")) == ""), "MISSING_CUSTOMER_ID")
     .when(~F.col("Currency").isin("USD", "EUR", "GBP"), "INVALID_CURRENCY")
     .when(F.col("ProductName").isNull(), "INVALID_PRODUCT_ID")
     .otherwise(None))

display(sales_enriched_df)

In [0]:
# Valid and Rejected Records

valid_df = sales_enriched_df.filter(F.col("ValidationError").isNull())
rejected_df = sales_enriched_df.filter(F.col("ValidationError").isNotNull())

display(valid_df)
display(rejected_df)

In [0]:
# Exchange Rate API

def get_exchange_rates():
    try:
        response = requests.get(API_URL, timeout=10)
        response.raise_for_status()
        data = response.json()

        usd_rate = float(data["rates"]["USD"])
        gbp_rate = float(data["rates"]["GBP"])

        return {"EUR": usd_rate, "GBP": usd_rate / gbp_rate, "USD": 1.0, "source": "ExchangeRate-API", "fallback": False}

    except Exception as e:
        print("Exchange API failed:", str(e))
        return {"EUR": 1.16, "GBP": 1.3567, "USD": 1.0, "source": "Default-Fallback", "fallback": True}

rates = get_exchange_rates()
print(rates)

In [0]:
# Convert Sale Amount to USD

valid_df = valid_df.withColumn("ConversionRate",
    F.when(F.col("Currency") == "USD", F.lit(rates["USD"]))
     .when(F.col("Currency") == "EUR", F.lit(rates["EUR"]))
     .when(F.col("Currency") == "GBP", F.lit(rates["GBP"]))
     .otherwise(None))

valid_df = valid_df.withColumn("SaleAmountUSD",
    F.round(F.col("SaleAmount") * F.col("ConversionRate"), 2))

display(valid_df.select("OrderID", "SaleAmount", "Currency", "ConversionRate", "SaleAmountUSD"))

In [0]:
# Currency Conversion Audit Log

conversion_log_df = valid_df.select("OrderID", "Currency", "ConversionRate") \
    .withColumn("ConversionTimestamp", F.current_timestamp()) \
    .withColumn("RateSource", F.lit(rates["source"])) \
    .withColumn("FallbackUsed", F.lit(rates["fallback"]))

display(conversion_log_df)

In [0]:
# Error Log and Rejected Records

rejected_records_df = rejected_df.withColumnRenamed("ValidationError", "ErrorType") \
    .withColumn("ErrorMessage", F.col("ErrorType")) \
    .withColumn("ErrorTimestamp", F.current_timestamp())

error_log_df = rejected_records_df.select("OrderID", "ProductID", "ErrorType", "ErrorMessage", "ErrorTimestamp")

display(error_log_df)

In [0]:
# Final Enriched Dataset

etl_run_id = str(__import__("uuid").uuid4())

final_df = valid_df.select(
    "OrderID",
    "ProductID",
    "ProductName",
    "Category",
    "SaleAmount",
    "Currency",
    "ConversionRate",
    "SaleAmountUSD",
    "OrderDate",
    "Region",
    "CustomerID",
    "Discount"
).withColumn("ProcessedDate", F.current_timestamp()) \
 .withColumn("ETLRunID", F.lit(etl_run_id))

display(final_df)

In [0]:
# Rejected Records Output

rejected_output_df = rejected_records_df.select(
    "OrderID",
    "ProductID",
    "SaleAmount",
    "OrderDate",
    "Region",
    "CustomerID",
    "Discount",
    "Currency",
    "ErrorType",
    "ErrorMessage",
    "ErrorTimestamp"
)

display(rejected_output_df)

In [0]:
# Save Clean Sales Data

final_df.write.mode("overwrite").option("header", True).csv(f"{OUTPUT_PATH}/clean_sales")

In [0]:
# Save Rejected Sales Data

rejected_output_df.write.mode("overwrite").option("header", True).csv(f"{OUTPUT_PATH}/rejected_sales")

In [0]:
# Save Currency Conversion Log

conversion_log_df.write.mode("overwrite").option("header", True).csv(f"{OUTPUT_PATH}/currency_conversion_log")

In [0]:
# Save Error Log

error_log_df.write.mode("overwrite").option("header", True).csv(f"{OUTPUT_PATH}/error_log")

In [0]:
# Final Data Quality Gate

total_records = sales_df.count()
valid_records = valid_df.count()
rejected_records = rejected_df.count()

rejection_rate = (rejected_records / total_records) * 100

print("Total records:", total_records)
print("Valid records:", valid_records)
print("Rejected records:", rejected_records)
print(f"Rejection rate: {rejection_rate:.2f}%")
print("Allowed threshold: 5%")

if rejection_rate > 5:
    print("Pipeline Status: FAILED")
    raise Exception(f"Data quality threshold exceeded. Rejected {rejection_rate:.2f}% of records; maximum allowed is 5%.")
else:
    print("Pipeline Status: SUCCESS")